# Lasso Regression — Statistics

**Goal.** Treat $\hat{\theta}_{lasso}$ as a random vector. Define the **active set** (the indices where $\hat{\theta}$ is non-zero), examine how Lasso shrinks active coefficients and zeros out inactive ones, sketch the celebrated **oracle inequalities** that guarantee Lasso's prediction error is close to that of a model that *knows* the true support, and turn all of this into practical $\lambda$-selection rules — k-fold CV and the **one-standard-error rule** of Hastie, Tibshirani, Friedman.

**Role of this notebook.** Estimator theory: math first, with minimal Monte Carlo simulations used only to *verify* a theorem or visualise an effect. We will not re-prove the oracle inequalities (they are graduate-level material) but will state them clearly enough to use as guidance.

**Prerequisites.** `01_linear_regression/04_statistics.ipynb` (the bias / variance framework, k-fold CV); `02_mathematics.ipynb` and `03_optimization.ipynb` of this folder.

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → **`04_statistics`** → `05_hands_on_programming`.

**Six questions.**

1. What is the **active set** of $\hat{\theta}_{lasso}$, and what determines it?
2. Why are the *selected* coefficients biased downward (shrunk)?
3. When does Lasso recover the true non-zero support? (Variable-selection consistency.)
4. What guarantee does Lasso give on *prediction* error? (Oracle inequalities.)
5. How do we choose $\lambda$ from data? (CV + the one-standard-error rule.)
6. What are the main *failure modes* of Lasso?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. The active set

### 1.1 Definition

Given a fitted Lasso estimate $\hat{\theta}_{lasso}$, define its **active set** (or **support**) as the indices of non-zero coordinates:

> A($\lambda$)  :=  { j $\in$ {1, …, p}  :  $\hat{\theta}_{lasso}$, j $\neq$ 0 }.

The active set is a *random* subset of {1, …, p} — it depends on the data y. Its size |A($\lambda$)| is the analogue of the effective dof for Ridge, but with a sharp discrete count rather than a continuous trace.

### 1.2 Inactive coordinates from first principles

Recall the KKT condition for inactive coordinates (Theorem 4.1 of `02_mathematics.ipynb`): $\theta_j$\* = 0 if and only if

> | (2 / n) $\cdot$ ⟨$x_j$, $r^*$⟩ |  $\le$  $\lambda$,    where $r^*$ = X $\theta^*$ - y.   (1.1)

Reading: a feature is kept only if its column $x_j$ has "strong enough" correlation with the residual at the optimum. As $\lambda$ grows the threshold rises, so more columns fail the condition and drop out — explaining why the active set shrinks monotonically with $\lambda$.

Two important limits:

- **$\lambda$ = 0.** Threshold is zero, every column passes ⇒ the active set is full, fit reduces to OLS.
- **$\lambda$ $\ge$ $\lambda_{\max}$ := (2 / n) $\cdot$ max_j |⟨$x_j$, y⟩|.** No column passes ⇒ the active set is empty, $\hat{\theta}_{lasso}$ = 0. This $\lambda_{\max}$ is exactly the upper end of the regularisation path used by every path solver (`03_optimization.ipynb` §5).

## 2. The bias of selected coefficients

Even after a feature is selected ($\hat{\theta}_{j}$ $\neq$ 0), its value is *not* the unregularised least-squares value: Lasso shrinks it toward zero.

### 2.1 Orthogonal-design closed form

When the columns of X are *orthogonal* ($X^T X$ = n $\cdot$ $I_p$), the soft-thresholding derivation (Theorem 4.2 of `02_mathematics.ipynb`) extends in one step to all coordinates simultaneously:

> $\hat{\theta}_{lasso}$, j  =  S_{$\lambda$ / 2}( $\hat{\theta}_{OLS}$, j ),    j = 1, …, p.   (2.1)

Each Lasso coordinate is exactly its OLS counterpart, *soft-thresholded*. So

- If |$\hat{\theta}_{OLS}$, j| $\le$ $\lambda$/2 then $\hat{\theta}_{lasso}$, j = 0 — *exact* selection.
- Otherwise $\hat{\theta}_{lasso}$, j = $\hat{\theta}_{OLS}$, j - sign($\hat{\theta}_{OLS}$, j) $\cdot$ ($\lambda$/2) — *shrunk by $\lambda$/2*.

### 2.2 Reading

The shrinkage `$\lambda$/2` is the same for *every* selected coordinate. Two consequences:

1. **Negative bias.** Selected coefficients are pulled toward zero by a constant amount.
2. **Bias does not shrink with n.** Unlike OLS (which is unbiased) and Ridge (whose bias shrinks as `1/n` for fixed $\lambda$ in the way the coordinate-by-coordinate formula scales), Lasso's bias for selected coefficients does not vanish even with infinite data — *unless* $\lambda$ → 0.

A practical fix: the **debiased Lasso** (or "two-stage Lasso"). Fit Lasso to select a subset; then fit OLS on only the selected features for an unbiased re-estimate of those magnitudes. This combines variable selection (from Lasso) with statistical optimality on the selected set (from OLS). Many modern Lasso pipelines do this.

### 2.3 Monte Carlo

Repeated draws of y, then look at the mean of $\hat{\theta}_{lasso}$, j and compare with the true $\theta_j$.

In [ ]:
# Use an orthogonal design so eq. (2.1) applies cleanly.
n, p = 200, 8
Q, _ = np.linalg.qr(rng.normal(size=(n, p)))
X = Q * np.sqrt(n)                               # columns now orthogonal with ‖x_j‖² = n

true_theta = np.array([3.0, -2.0, 1.5, 0.0, 0.0, 1.0, 0.0, 0.0])
sigma = 0.5
lam = 0.6                                        # our convention; sklearn uses alpha = lam / 2
alpha_skl = lam / 2

M = 300
estimates = np.empty((M, p))
for m in range(M):
    y = X @ true_theta + rng.normal(0, sigma, size=n)
    estimates[m] = Lasso(alpha=alpha_skl, fit_intercept=False, max_iter=20000).fit(X, y).coef_

print(f"λ (our convention) = {lam}    → predicted shrinkage λ/2 = {lam/2}")
print()
print(f"{'j':>3}  {'true θⱼ':>10}  {'mean θ̂_lasso,j':>15}  {'bias':>10}  {'P(j selected)':>15}")
for j in range(p):
    mean_hat = estimates[:, j].mean()
    bias     = mean_hat - true_theta[j]
    p_select = float(np.mean(np.abs(estimates[:, j]) > 1e-8))
    print(f"{j:>3}  {true_theta[j]:>10.4f}  {mean_hat:>15.4f}  {bias:>10.4f}  {p_select:>15.2f}")

**Reading.**

- The four truly non-zero features (indices 0, 1, 2, 5) are selected almost every time and their mean estimate is biased *toward zero* by roughly `$\lambda$ / 2 $\approx$ 0.15` — the orthogonal-design prediction (2.1).
- The four irrelevant features (3, 4, 6, 7) are dropped most of the time, but their mean is non-zero — occasionally a noise spike pushes the empirical correlation across the threshold and the feature *enters* the active set with a small noise-driven coefficient.
- P(j selected) is the analogue of "detection probability" for variable selection. For irrelevant features, this is the false-positive rate.

## 3. Variable-selection consistency (a glimpse)

A natural question: under what conditions does the Lasso active set A($\lambda$) match the *true* support `S := {j : $\theta_{j,\mathrm{true}}$ $\neq$ 0}` exactly, with high probability, as n grows?

This is **variable-selection consistency** or **support recovery**. The full theory is technical (Zhao & Yu 2006, Meinshausen & Bühlmann 2006); the upshot is two conditions:

1. **Sample size large enough relative to sparsity.** Roughly, n must scale like `s $\cdot$ log p`, where `s = |S|` is the number of true non-zero coefficients. So you can have `p ≫ n`, as long as the true model is sparse.
2. **Irrepresentable condition.** Loosely: the *irrelevant* features (columns outside S) must not be too well represented by linear combinations of the *relevant* features (columns inside S). Formally, there is a quantity `(1/n) $\cdot$ X_{Sᶜ}ᵀ X_S $\cdot$ (1/n $\cdot$ X_Sᵀ X_S)^{-1}` whose every entry must be bounded by 1 in absolute value, with strict slack.

When the irrepresentable condition fails — typically when features are *highly correlated* — Lasso may pick the wrong representative from a correlated group, or sample-to-sample swap between equivalent features. This is the main practical weakness of Lasso, and motivates the **Elastic Net** (a hybrid L1 + L2 penalty; see Zou & Hastie 2005) which is more stable on correlated designs.

**Reading.** The mathematics is deep, but the practical takeaway is simple:

- If features are roughly independent and the true model is sparse, Lasso recovers the support.
- If features are highly correlated and grouped, Lasso may pick one representative and drop the rest. The *prediction* is still good; the *interpretation* may be misleading.
- When in doubt, use Elastic Net (next algorithm folder after this one, conceptually) or stability selection (Meinshausen & Bühlmann 2010), which subsamples and refits Lasso many times and reports features selected on most subsamples.

## 4. Oracle inequalities for prediction error

Even when Lasso does *not* recover the exact support, it gives a strong guarantee on *prediction* error. The theorem below is a stripped-down version of a classical result (Bickel, Ritov, Tsybakov 2009, *Annals of Statistics*).

### 4.1 Theorem (oracle inequality, schematic)

> **Theorem 4.1 (schematic).** Under mild regularity on X (a restricted-eigenvalue condition) and Gaussian noise with variance $\sigma^2$, if $\lambda$ is chosen on the order of $\sigma \sqrt{\log p / n}$, then with high probability,
>
> (1 / n) $\cdot$ $\|X(\hat{\theta}_{\mathrm{lasso}} - \theta_{\mathrm{true}})\|_2^2$   $\le$   C $\cdot$ $\sigma^2$ $\cdot$ s $\cdot$ log p / n,
>
> where `s := |S|` is the number of truly non-zero coefficients and C is an absolute constant.

**Reading.** The right side is *almost* what an oracle would achieve. The hypothetical oracle that *knew* the true support would fit OLS on those s columns and pay prediction error of order `$\sigma^2$ $\cdot$ s / n` — this is the Gauss–Markov benchmark on the right model. Lasso pays the same `$\sigma^2$ $\cdot$ s / n` plus an extra factor `log p` — the price of not knowing which s features matter.

**The log p factor is the right cost.** In information-theoretic terms, identifying s features out of p requires `log( C(p, s) ) $\approx$ s $\cdot$ log(p/s)` bits of information; the corresponding `log p / n` factor in the rate is unavoidable.

**Take-home.** Lasso achieves *near-oracle* prediction performance, even when p is much larger than n, as long as the truth is sparse. This is the central theoretical reason Lasso (and its generalisations) is the workhorse for high-dimensional regression.

## 5. Choosing $\lambda$ in practice

Same k-fold CV as Ridge and polynomial regression, with one twist that is specific to Lasso (and Ridge): the **one-standard-error rule** (1-SE rule).

### 5.1 The 1-SE rule

Let CV($\lambda$) be the k-fold cross-validation error at penalty $\lambda$, and SE($\lambda$) be the standard error of that estimate across folds. Let $\lambda_{\min}$ be the minimiser of CV($\lambda$). The **1-SE $\lambda$** is

> $\lambda_{1SE}$  :=  max { $\lambda$  :  CV($\lambda$)  $\le$  CV($\lambda_{\min}$)  +  SE($\lambda_{\min}$) }.   (5.1)

In words: among all $\lambda$ whose CV error is within one standard error of the minimum, choose the *largest*. This intentionally picks a *more regularised* (sparser, more stable) model, accepting a small loss in average CV error for a large gain in interpretability and out-of-sample stability.

The 1-SE rule is the recommended default in Hastie, Tibshirani, Friedman *Elements of Statistical Learning*. Most CV-based Lasso pipelines report both $\lambda_{\min}$ and $\lambda_{1SE}$; the user picks one depending on whether they want the lowest-CV model or the simplest reasonable model.

### 5.2 Algorithm

```
ALGORITHM:  k-fold CV with the 1-SE rule

1.  For each $\lambda$ in a grid (built by the path solver of 03_optimization.ipynb §5):
       compute err_j($\lambda$) on fold j, for j = 1, ..., k
       CV($\lambda$)  $\leftarrow$  mean_j err_j($\lambda$)
       SE($\lambda$)  $\leftarrow$  stddev_j err_j($\lambda$) / $\sqrt{k}$

2.  $\lambda_{\min}$   $\leftarrow$  $\arg\min_{\lambda}$  CV($\lambda$)

3.  $\lambda_{1SE}$   $\leftarrow$  largest $\lambda$ with  CV($\lambda$) $\le$ CV($\lambda_{\min}$) + SE($\lambda_{\min}$)

4.  Refit Lasso on the full data at  $\lambda_{\min}$  (or $\lambda_{1SE}$) and return.
```

### 5.3 Demo

In [ ]:
# Sparse synthetic problem; run 5-fold CV across a λ grid.
n, p = 250, 50
X = rng.normal(size=(n, p))
true_theta = np.zeros(p); true_theta[:6] = [2.5, -2.0, 1.5, 1.0, -0.8, 0.5]
y = X @ true_theta + rng.normal(0, 0.4, size=n)

X = X - X.mean(axis=0); y = y - y.mean()

lams = np.logspace(-3, 0, 40)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

fold_err = np.zeros((len(lams), 5))
for f, (tr_idx, te_idx) in enumerate(kf.split(X)):
    for i, lam in enumerate(lams):
        m = Lasso(alpha=lam, fit_intercept=False, max_iter=20000).fit(X[tr_idx], y[tr_idx])
        fold_err[i, f] = np.mean((y[te_idx] - m.predict(X[te_idx])) ** 2)

cv  = fold_err.mean(axis=1)
se  = fold_err.std(axis=1) / np.sqrt(5)

i_min  = int(np.argmin(cv))
lam_min = lams[i_min]
thresh  = cv[i_min] + se[i_min]
# largest λ with CV ≤ thresh
i_1se  = int(np.max(np.where(cv <= thresh)[0]))
lam_1se = lams[i_1se]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.errorbar(lams, cv, yerr=se, fmt="o-", color="crimson", lw=1, markersize=3)
ax.axvline(lam_min, color="black",   ls="--", label=f"λ_min   = {lam_min:.3f}")
ax.axvline(lam_1se, color="seagreen", ls="--", label=f"λ_1SE   = {lam_1se:.3f}")
ax.axhline(thresh,  color="black",   ls=":",  lw=1, label="CV(λ_min) + SE")
ax.set_xscale("log")
ax.set_xlabel("λ (log)")
ax.set_ylabel("k-fold CV MSE  (mean ± SE)")
ax.set_title("5-fold CV path with the 1-SE rule")
ax.legend()
plt.show()

# Compare # selected features at both λ values.
model_min = Lasso(alpha=lam_min, fit_intercept=False, max_iter=20000).fit(X, y)
model_1se = Lasso(alpha=lam_1se, fit_intercept=False, max_iter=20000).fit(X, y)
print(f"true non-zero    = {int(np.sum(true_theta != 0))}")
print(f"at λ_min  (= {lam_min:.3f}):  selected = {int(np.sum(np.abs(model_min.coef_) > 1e-8))}")
print(f"at λ_1SE  (= {lam_1se:.3f}):  selected = {int(np.sum(np.abs(model_1se.coef_) > 1e-8))}")

**Reading.** The 1-SE rule trades a tiny amount of CV error for a substantially sparser model. On this synthetic problem the truth has 6 non-zero features; $\lambda_{\min}$ usually keeps more (some noise features sneak in), while $\lambda_{1SE}$ is closer to the true sparsity. This is the practical reason the 1-SE rule is preferred when *interpretation* matters.

## 6. Failure modes — when to *not* use Lasso

Lasso is the right tool for *most* high-dimensional regression problems, but it has predictable failure modes:

1. **Correlated features.** If two features carry the same signal, Lasso typically picks one and drops the other — and the choice is sample-dependent. Selected coefficients are unstable; predictions are still fine. *Cure:* Elastic Net (L1 + L2 penalty mix), or post-Lasso refit on the union of supports across CV folds.
2. **Grouped variables.** Same issue at scale — when a true effect is spread across many correlated features (e.g. dummy-encoded categorical variables), Lasso picks an arbitrary representative. *Cure:* group Lasso (penalty on the L2 norm of each group, not the L1 of all entries).
3. **Strong non-sparse truth.** If many features are weakly relevant (no zeros in $\theta_{true}$), Lasso's exact-zero behaviour hurts. *Cure:* Ridge, or Elastic Net biased toward L2.
4. **Heteroscedastic / non-Gaussian noise.** The oracle inequality (§4) assumes sub-Gaussian noise; with heavy tails Lasso's $\lambda$ should be inflated or replaced with a *robust* version (quantile Lasso, Huber Lasso).
5. **Logistic / non-quadratic loss.** The same penalty plugs into logistic regression and Poisson regression — but the optimisation step changes. `sklearn.linear_model.LogisticRegression(penalty="l1")` handles this; see `05_logistic_regression/` later in this module.

## Takeaway

- **Active set (§1).** `A($\lambda$) = {j : $\hat{\theta}_{j}$ $\neq$ 0}`. A coordinate is inactive iff `|⟨$x_j$, $r^*$⟩| $\le$ n $\lambda$ / 2`. The maximum useful $\lambda$ is `$\lambda_{\max}$ = (2/n) $\cdot$ max_j |⟨$x_j$, y⟩|`.
- **Shrinkage bias (§2).** For orthogonal designs, `$\hat{\theta}_{lasso}$, j = S_{$\lambda$/2}($\hat{\theta}_{OLS}$, j)` (eq. 2.1). Selected coefficients are pulled toward zero by `$\lambda$/2`. Bias does *not* vanish with n.
- **Support recovery (§3).** Requires sample size `n ≳ s $\cdot$ log p` and the **irrepresentable condition** (relevant features not too well-explained by irrelevant ones). Fails when features are highly correlated.
- **Oracle inequality (§4).** `(1/n) $\cdot$ $\|X(\hat{\theta} - \theta_{\mathrm{true}})\|^2$ $\le$ C $\sigma^2$ $\cdot$ s $\cdot$ log p / n` with high probability, when $\lambda$ is on the order of $\sigma \sqrt{\log p / n}$. Lasso is near-oracle for prediction even when p ≫ n.
- **$\lambda$-selection (§5).** k-fold CV + the **1-SE rule** for a sparser, more stable model: largest $\lambda$ within one SE of the CV minimum.
- **Failure modes (§6).** Correlated features → unstable support → use Elastic Net. Non-sparse truth → use Ridge.

Next: `05_hands_on_programming.ipynb` — implement coordinate descent from scratch, build a CV path solver with the 1-SE rule, cross-check against `sklearn.linear_model.LassoCV`, and demonstrate variable selection on the diabetes dataset.